# 01 · Tratamento e preparação dos dados

**Tech Challenge Fase 1 · NPS Preditivo**

Este notebook cobre as etapas de *entendimento* e *preparação* dos dados do CRISP-DM. Antes de
qualquer gráfico ou modelo, a pergunta é: **dá para confiar nesta base?**

Roteiro:

1. leitura da base bruta;
2. validação do esquema (as colunas do dicionário de dados estão todas lá?);
3. diagnóstico de qualidade (nulos, duplicados, tipos);
4. valores fora da faixa possível;
5. regras de consistência entre colunas;
6. limpeza;
7. análise de sensibilidade (as linhas suspeitas mudam alguma conclusão?);
8. colunas de apoio para a análise;
9. gravação da base tratada.

As funções usadas aqui estão em [`src/data_prep.py`](../src/data_prep.py). O mesmo tratamento roda
fora do notebook com `python -m src.data_prep`, e por isso o resultado é sempre o mesmo.

In [1]:
import sys
from pathlib import Path

# Localiza a raiz do projeto (a pasta que contém src/) para importar o código compartilhado,
# não importa de onde o notebook seja aberto.
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src import config, data_prep

pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

## 1. Leitura da base bruta

O arquivo `desafio_nps_fase_1.csv` foi baixado do repositório do curso
([POSTECH_AI_SCIENTIST](https://github.com/AnaRaquelCafe/POSTECH_AI_SCIENTIST/tree/main/Base%20de%20dados%20Tech%20Challenge))
e está em `data/raw/` sem nenhuma alteração.

In [2]:
raw_df = data_prep.load_raw_data()
print(f"{raw_df.shape[0]:,} linhas x {raw_df.shape[1]} colunas")
raw_df.head()

2,500 linhas x 19 colunas


,customer_id,customer_age,customer_region,customer_tenure_months,order_id,order_value,items_quantity,discount_value,payment_installments,delivery_time_days,delivery_delay_days,freight_value,delivery_attempts,customer_service_contacts,resolution_time_days,nps_score,repeat_purchase_30d,complaints_count,csat_internal_score
0,1,63,Nordeste,14,50001,139.73,4,39.35,4,2,2,55.53,3,0,4,6.90,0,3,6.50
1,2,20,Sul,1,50002,458.95,2,9.51,10,6,4,28.23,3,0,10,2.40,0,3,0.00
2,3,46,Nordeste,111,50003,507.06,5,42.82,6,6,1,40.99,1,4,5,4.80,0,7,1.50
3,4,52,Centro-Oeste,117,50004,302.19,2,19.58,9,5,2,35.24,3,1,11,5.90,0,4,0.30
4,5,56,Norte,50,50005,253.06,1,29.37,11,13,1,39.32,1,1,0,6.10,0,3,7.90


## 2. Validação do esquema

Se alguma coluna do dicionário de dados estiver faltando, o processo para aqui com um erro claro,
em vez de falhar lá na frente com uma mensagem difícil de entender.

In [3]:
data_prep.validate_schema(raw_df)
print(f"Esquema OK: as {len(config.EXPECTED_COLUMNS)} colunas esperadas estão presentes.")

Esquema OK: as 19 colunas esperadas estão presentes.


## 3. Diagnóstico de qualidade

In [4]:
data_prep.quality_report(raw_df)

,tipo,nulos,pct_nulos,valores_distintos,minimo,maximo
coluna,,,,,,
complaints_count,int64,0,0.00,12,0.00,11.00
csat_internal_score,float64,0,0.00,98,0.00,10.00
customer_age,int64,0,0.00,52,18.00,69.00
customer_id,int64,0,0.00,2500,1.00,"2,500.00"
customer_region,str,0,0.00,5,NaN,NaN
customer_service_contacts,int64,0,0.00,8,0.00,7.00
customer_tenure_months,int64,0,0.00,119,1.00,119.00
delivery_attempts,int64,0,0.00,3,1.00,3.00
delivery_delay_days,int64,0,0.00,9,0.00,8.00


In [5]:
print("Linhas inteiramente duplicadas:", raw_df.duplicated().sum())
print("Pedidos (order_id) repetidos:  ", raw_df["order_id"].duplicated().sum())
print("Clientes (customer_id) repetidos:", raw_df["customer_id"].duplicated().sum())

share_integer_scores = (raw_df["nps_score"] % 1 == 0).mean()
print(f"\nNotas de NPS que são números inteiros: {share_integer_scores:.0%}")
print("Exemplos de notas:", sorted(raw_df["nps_score"].sample(8, random_state=1).tolist()))

Linhas inteiramente duplicadas: 0
Pedidos (order_id) repetidos:   0
Clientes (customer_id) repetidos: 0

Notas de NPS que são números inteiros: 17%
Exemplos de notas: [0.0, 0.6, 2.1, 4.2, 5.2, 5.6, 6.3, 8.3]


**O que encontramos:**

- **Nenhum valor nulo** e **nenhuma linha duplicada**.
- **Cada cliente aparece uma única vez** (um pedido por cliente). Por isso não dá para acompanhar a
  evolução do mesmo cliente ao longo do tempo: cada linha é uma "fotografia" de uma jornada.
- **A nota de NPS tem casa decimal** (6,9; 2,4...). Só 17% das notas são inteiras. Na pesquisa
  clássica o cliente escolhe um número inteiro de 0 a 10, então essa nota provavelmente já passou
  por algum tratamento (média de respostas ou ajuste interno). Para classificar detratores,
  neutros e promotores usamos a **parte inteira da nota**: 6,9 continua sendo detrator (abaixo de 7).
  Essa decisão está registrada em `src/config.py`.

## 4. Valores fora da faixa possível

Cada coluna tem uma faixa que faz sentido no mundo real (NPS entre 0 e 10, quantidade de itens
pelo menos 1, nenhum valor de contagem negativo...). Valor fora dessa faixa é erro de registro.

In [6]:
range_violations = data_prep.find_range_violations(raw_df)
pd.DataFrame(
    {
        "faixa_valida": {column: data_prep.VALID_RANGES[column] for column in range_violations.columns},
        "linhas_fora_da_faixa": range_violations.sum(),
    }
)

,faixa_valida,linhas_fora_da_faixa
customer_age,"(16, 110)",0
customer_tenure_months,"(0, None)",0
order_value,"(0, None)",0
items_quantity,"(1, None)",0
discount_value,"(0, None)",0
payment_installments,"(1, 24)",0
delivery_time_days,"(0, None)",0
delivery_delay_days,"(0, None)",0
freight_value,"(0, None)",0
delivery_attempts,"(1, None)",0


In [7]:
raw_df.drop(columns=config.ID_COLUMNS).describe().T

,count,mean,std,min,25%,50%,75%,max
customer_age,"2,500.00",43.40,14.89,18.00,31.00,43.00,56.00,69.00
customer_tenure_months,"2,500.00",61.32,34.48,1.00,31.00,62.00,91.00,119.00
order_value,"2,500.00",434.26,289.77,7.76,220.25,375.51,577.29,"1,983.81"
items_quantity,"2,500.00",3.47,1.69,1.00,2.00,3.00,5.00,6.00
discount_value,"2,500.00",29.75,29.23,0.02,8.88,20.94,40.83,230.33
payment_installments,"2,500.00",6.00,3.16,1.00,3.00,6.00,9.00,11.00
delivery_time_days,"2,500.00",8.02,3.77,2.00,5.00,8.00,11.00,14.00
delivery_delay_days,"2,500.00",2.19,1.45,0.00,1.00,2.00,3.00,8.00
freight_value,"2,500.00",38.22,12.08,2.62,29.93,38.50,46.27,76.13
delivery_attempts,"2,500.00",2.01,0.82,1.00,1.00,2.00,3.00,3.00


Nenhum valor impossível. Os mínimos e máximos são plausíveis para um e-commerce: pedidos de
R\$ 7,76 a R\$ 1.983,81; entregas de 2 a 14 dias; atrasos de 0 a 8 dias; clientes de 18 a 69 anos.

## 5. Regras de consistência entre colunas

Olhar coluna por coluna não basta: alguns problemas só aparecem quando uma coluna é comparada
com outra. Testamos três **inconsistências** (combinações que não deveriam existir) e dois
**pontos de atenção** (combinações possíveis, mas que pedem confirmação com a área de negócio).

In [8]:
data_prep.consistency_summary(raw_df)

,regra,tipo,coluna_flag,linhas,pct_linhas
0,Atraso maior que o prazo total de entrega,inconsistência,flag_delay_exceeds_delivery_time,121,4.80
1,Desconto maior ou igual ao valor do pedido,inconsistência,flag_discount_exceeds_order_value,35,1.40
2,Tempo de casa maior que a vida adulta do cliente,inconsistência,flag_tenure_exceeds_adult_life,263,10.50
3,Reclamação registrada sem contato com o atendi...,ponto de atenção,flag_complaint_without_contact,531,21.20
4,Tempo de resolução registrado sem contato com ...,ponto de atenção,flag_resolution_without_contact,511,20.40


In [9]:
share_complaints_above_contacts = (raw_df["complaints_count"] > raw_df["customer_service_contacts"]).mean()
print(f"Pedidos com mais reclamações do que contatos com o atendimento: {share_complaints_above_contacts:.0%}")

Pedidos com mais reclamações do que contatos com o atendimento: 97%


**Como interpretar:**

| Regra | Leitura | Decisão |
|---|---|---|
| Atraso maior que o prazo total (4,8%) | Se o prazo total já inclui o atraso, isso não deveria acontecer. Pode ser que o prazo seja contado a partir de outro marco (ex.: postagem) | Manter e sinalizar |
| Desconto ≥ valor do pedido (1,4%) | Pedido "pago" inteiramente com cupom ou erro de cadastro do desconto | Manter e sinalizar |
| Tempo de casa maior que a vida adulta (10,5%) | Ex.: cliente de 20 anos com 9 anos de casa. Pode ser conta criada por um responsável | Manter e sinalizar |
| Reclamação sem contato com o atendimento (21%) | Reclamações podem vir de outros canais (marketplace, redes sociais, Reclame Aqui) | Ponto de atenção |
| Resolução registrada sem contato (20%) | Problema resolvido sem o cliente ter acionado o atendimento | Ponto de atenção |

Além disso, **97% dos pedidos têm mais reclamações do que contatos**. Isso sugere que
`complaints_count` conta reclamações **de todos os canais e/ou do histórico do cliente**, e não só
deste pedido. É uma dúvida que precisa ser levada à área dona do dado antes de usar o modelo em
produção.

**Por que não remover essas linhas?** Elas somam 16% da base. Apagar 1 em cada 6 pedidos sem saber
se o problema é do dado ou da nossa interpretação seria jogar fora informação. Em vez disso, cada
regra vira uma coluna `flag_*`, e no passo 7 medimos se essas linhas mudam alguma conclusão.

## 6. Limpeza

Critério: **só sai da base o que não pode ser usado de jeito nenhum** (duplicado, sem nota de NPS
ou com valor impossível). Nulos nas variáveis de entrada não são preenchidos aqui: isso fica dentro
do pipeline do modelo, aprendido só com os dados de treino, para não vazar informação do teste.

In [10]:
clean_df, cleaning_log = data_prep.clean_data(raw_df)
pd.Series(cleaning_log, name="linhas")

linhas_iniciais                      2500
removidas_duplicadas                    0
removidas_sem_nps                       0
removidas_valor_impossivel              0
linhas_finais                        2500
linhas_sinalizadas_inconsistentes     405
Name: linhas, dtype: int64

## 7. Análise de sensibilidade: as linhas suspeitas mudam alguma conclusão?

Comparamos a relação de cada fator com a nota (correlação de Spearman) usando **todas as linhas**
e **só as linhas sem inconsistência**. Se os números forem parecidos, manter as linhas é seguro.

In [11]:
key_columns = config.OPERATIONAL_DRIVERS + ["delivery_time_days", "order_value", "customer_age"]
consistent_rows = clean_df["has_inconsistency"] == 0

sensitivity = pd.DataFrame(
    {
        "todas_as_linhas": clean_df[key_columns].corrwith(clean_df["nps_score"], method="spearman"),
        "sem_inconsistencias": clean_df.loc[consistent_rows, key_columns].corrwith(
            clean_df.loc[consistent_rows, "nps_score"], method="spearman"
        ),
    }
)
sensitivity["diferenca"] = sensitivity["sem_inconsistencias"] - sensitivity["todas_as_linhas"]
sensitivity.index = sensitivity.index.map(config.FEATURE_LABELS)

print(f"NPS com todas as linhas:       {data_prep.nps_index(clean_df['nps_score'])}")
print(f"NPS sem linhas inconsistentes: {data_prep.nps_index(clean_df.loc[consistent_rows, 'nps_score'])}")
sensitivity.round(3)

NPS com todas as linhas:       -80.0
NPS sem linhas inconsistentes: -78.8


,todas_as_linhas,sem_inconsistencias,diferenca
Dias de atraso,-0.59,-0.58,0.01
Reclamações,-0.49,-0.50,-0.01
Contatos com o atendimento,-0.34,-0.34,-0.01
Tempo de resolução (dias),-0.19,-0.19,0.00
Prazo total de entrega (dias),0.00,-0.07,-0.07
Valor do pedido (R$),0.04,0.03,-0.01
Idade do cliente,-0.01,-0.01,0.00


As diferenças ficam na segunda casa decimal e nenhum fator muda de importância. O NPS passa de
-80,0 para -78,8. **As conclusões não dependem das linhas suspeitas**, então elas ficam na base
(sinalizadas).

## 8. Colunas de apoio para a análise

Para conversar com a área de negócio, criamos colunas que traduzem números em grupos fáceis de ler.
**Nenhuma delas entra no modelo**: servem só para a análise exploratória.

| Coluna | O que é |
|---|---|
| `nps_category` | detrator (0 a 6), neutro (7 a 8) ou promotor (9 a 10) |
| `is_detractor`, `is_promoter` | 1/0, facilitam calcular percentuais |
| `on_time` | 1 se o pedido chegou sem atraso |
| `net_order_value`, `discount_pct` | valor pago depois do desconto e % de desconto |
| `delay_range`, `complaints_range`, `contacts_range`, `resolution_range` | faixas dos fatores operacionais. Os cortes vieram de uma árvore de decisão rasa (notebook 02) |
| `delivery_time_range`, `age_range`, `tenure_range`, `order_value_range`, `installments_range` | faixas de prazo, perfil e pedido |
| `experience_profile` | experiência **Boa**, **Intermediária** ou **Ruim**, combinando atraso, reclamações e contatos |

In [12]:
processed_df = data_prep.add_analysis_columns(clean_df)
processed_df[["nps_score", "nps_category", "delivery_delay_days", "delay_range",
              "complaints_count", "complaints_range", "experience_profile"]].head(8)

,nps_score,nps_category,delivery_delay_days,delay_range,complaints_count,complaints_range,experience_profile
0,6.90,detrator,2,1 a 2 dias,3,3 a 4,Intermediária
1,2.40,detrator,4,4 dias ou mais,3,3 a 4,Ruim
2,4.80,detrator,1,1 a 2 dias,7,5 ou mais,Ruim
3,5.90,detrator,2,1 a 2 dias,4,3 a 4,Intermediária
4,6.10,detrator,1,1 a 2 dias,3,3 a 4,Intermediária
5,0.90,detrator,5,4 dias ou mais,5,5 ou mais,Ruim
6,1.40,detrator,3,3 dias,6,5 ou mais,Ruim
7,0.00,detrator,5,4 dias ou mais,2,2,Ruim


In [13]:
category_share = processed_df["nps_category"].value_counts(normalize=True).mul(100).round(1)
print(category_share.to_string())
print(f"\nNPS da base (% promotores - % detratores): {data_prep.nps_index(processed_df['nps_score'])}")

nps_category
detrator   84.40
neutro     11.20
promotor    4.40

NPS da base (% promotores - % detratores): -80.0


## 9. Gravação da base tratada

In [14]:
config.DATA_PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
processed_df.to_csv(config.DATA_PROCESSED_PATH, index=False, encoding="utf-8")
print(f"Base tratada salva em {config.DATA_PROCESSED_PATH.relative_to(config.PROJECT_ROOT)}")
print(f"{processed_df.shape[0]:,} linhas x {processed_df.shape[1]} colunas")

Base tratada salva em data\processed\nps_tratado.csv
2,500 linhas x 41 colunas


## Resumo do tratamento

- A base tem **2.500 pedidos, um por cliente**, sem nulos, sem duplicados e sem valores impossíveis.
- **405 linhas (16%) têm alguma inconsistência entre colunas.** Foram mantidas e sinalizadas,
  porque a análise de sensibilidade mostrou que elas não mudam nenhuma conclusão.
- **Dúvidas para levar à área dona do dado:** o que exatamente `complaints_count` conta, a partir
  de qual marco o prazo de entrega é medido e por que a nota de NPS tem casa decimal.
- A base tratada (`data/processed/nps_tratado.csv`) é o ponto de partida dos notebooks 02 e 03.